In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)

In [ ]:
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"


In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

def clean_text(text):
  text = text.lower()
  text = re.sub(r'\d{5,}', ' longnumber ', text)
  text = text.replace('£', ' poundsign ')
  return text

train_messages = tf.constant([clean_text(m) for m in train_df['message']])
test_messages = tf.constant([clean_text(m) for m in test_df['message']])

train_labels = (train_df['label'] == 'spam').astype('float32').values
test_labels = (test_df['label'] == 'spam').astype('float32').values

In [ ]:
vectorizer = layers.TextVectorization(max_tokens=10000,
                                      output_sequence_length=60,
                                      standardize='lower_and_strip_punctuation')
vectorizer.adapt(train_messages)

model = keras.Sequential([
    layers.Input(shape=(), dtype='string'),
    vectorizer,
    layers.Embedding(10000, 32),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss',
                                           patience=5,
                                           restore_best_weights=True)

history = model.fit(train_messages,
                    train_labels,
                    epochs=40,
                    validation_data=(test_messages, test_labels),
                    callbacks=[early_stop],
                    class_weight={0: 1.0, 1: 4.0},
                    verbose=2)

In [ ]:
def predict_message(pred_text):
  probability = float(model.predict(tf.constant([clean_text(pred_text)]), verbose=0)[0][0])
  label = 'spam' if probability > 0.5 else 'ham'
  return [probability, label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()